In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Evaluate a Conversational Analytics (Gemini Data Analytics) data agent

This notebook builds an evaluation dataset for a BigQuery Conversational
Analytics data agent and scores the agent with the Vertex AI (Agent Platform)
Gen AI evaluation service.

- **Section A** builds an `EvaluationSet` of grounded `(question, golden_sql)`
  pairs: synthetic generation, validation on BigQuery, and a review step.
- **Section B** submits an `EvaluationRun` against your data agent and scores
  every answer with the `bq_ca_data_result_accuracy_v1` metric.

The `EvaluationSet` from Section A flows straight into Section B.

### Two APIs, two locations

This notebook calls **two different APIs**, each with its own location:

| Setting | API | Meaning | Typical value |
|---|---|---|---|
| `DATA_AGENT_LOCATION` | Gemini Data Analytics (`geminidataanalytics`) | where the **data agent** lives | `global` |
| `REGION` | Vertex AI (Agent Platform) | where the **EvaluationSet / EvaluationRun** live | `us-central1` |

They are independent and do not need to match. A `global` data agent evaluated
from `us-central1` is the normal setup. `DATA_AGENT_LOCATION` also supports
multi-regions (`us`, `eu`) and regional agents (see
[Conversational Analytics API locations](https://cloud.google.com/gemini/docs/conversational-analytics-api/locations)).
`REGION` must be a Vertex AI (Agent Platform) region (for example
`us-central1`); `global` is not supported for evaluation.

### Projects and identities

- **`EVAL_PROJECT`**: holds the `EvaluationSet` and `EvaluationRun`.
- **`DATA_AGENT_PROJECT`**: holds the data agent and its BigQuery data. It can
  be the same as `EVAL_PROJECT`.
- **`SERVICE_ACCOUNT`**: a service account **in `EVAL_PROJECT`** that you own.
  The Vertex AI (Agent Platform) Service Agent of `EVAL_PROJECT` impersonates it
  to chat with the data agent, and the agent runs its BigQuery SQL as this
  account.
- **`STAGING_BUCKET`**: a Cloud Storage bucket for the dataset file and the run
  outputs (when `USE_GCS_IMPORT = True`).

### Costs

Running this notebook bills your Google Cloud project(s) for the following
services:

- **BigQuery**: Grounding scans in Step 1 (capped at `MAX_GB_BILLED` per table),
  golden SQL validation queries in Step 5 (10 concurrent workers, up to
  `MAX_REPAIRS` repair attempts per item, each capped at `MAX_GB_BILLED`), and
  BigQuery queries executed by your data agent during the `EvaluationRun`
  (billed to `DATA_AGENT_PROJECT`). See
  [BigQuery pricing](https://cloud.google.com/bigquery/pricing).
- **Vertex AI / Gemini Enterprise Agent Platform**: Gemini model tokens for
  synthetic data generation (`DataFoundryService` in Step 4), SQL repair calls
  (`REPAIR_MODEL` in Step 5), data agent chat turns, and evaluation judge tokens
  for scoring with `bq_ca_data_result_accuracy_v1` in Step 9. See
  [Vertex AI pricing](https://cloud.google.com/vertex-ai/generative-ai/pricing) and [Gemini Enterprise Agent Platform pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing).
- **Cloud Storage**: Storing the staged dataset JSONL file and per-item
  evaluation run outputs in `STAGING_BUCKET` (when `USE_GCS_IMPORT = True`). See
  [Cloud Storage pricing](https://cloud.google.com/storage/pricing).

Run the notebook **top to bottom**. Start with the **prerequisite cell** (IAM
grants). The one step that needs your judgment is reviewing the generated
golden SQL queries (Section A, step 6) before you register them.

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/gda_agent_evaluation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/gemini_data_analytics/gda_agent_evaluation.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/gda_agent_evaluation.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

In [ ]:
# @title Install dependencies
%pip install --quiet --upgrade \
    "google-cloud-aiplatform" \
    "google-cloud-geminidataanalytics" \
    "google-cloud-bigquery" \
    "google-genai" \
    "db-dtypes" "pandas" "tabulate"

In [ ]:
# @title Imports + auth
import collections
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime
import json
import math
import random
import re
import time
import urllib.parse
import uuid

from google import genai
from google.api_core import client_options as client_options_lib
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import aiplatform_v1
from google.cloud import bigquery
from google.cloud import storage
import pandas as pd
import requests

try:
  from google.cloud import geminidataanalytics_v1 as geminidataanalytics
except ImportError:
  from google.cloud import geminidataanalytics_v1beta as geminidataanalytics

try:
  from google.colab import auth as _colab_auth

  _colab_auth.authenticate_user()
  print("Colab auth OK.")
except ImportError:
  print("Not in Colab -- using ambient Application Default Credentials.")

In [ ]:
# @title Configuration (edit me)
# ===========================================================================
# DATA_AGENT_LOCATION vs REGION -- two different APIs. NOT interchangeable.
#
#   DATA_AGENT_LOCATION   geminidataanalytics API: where the data agent
#                         RESOURCE lives. Almost always "global". Set a
#                         multi-region ("us", "eu") or supported single region
#                         only if the agent was created there; step 1 then
#                         routes to geminidataanalytics.{loc}.rep.googleapis.com
#                         or geminidataanalytics-{loc}.googleapis.com.
#
#   REGION                Vertex AI (Agent Platform) API: where the EvaluationSet
#                         and EvaluationRun live, and the endpoint hostname built
#                         below. MUST be a real region -- "global" is not
#                         supported for evaluation runs.
#
# A "global" agent evaluated from "us-central1" is the normal, expected setup.
# The two values are not supposed to match.
# ===========================================================================

# @markdown ### 1a. The agent under test — *geminidataanalytics* API
# @markdown Where the data agent and its BigQuery tables live.
# @markdown `DATA_AGENT_LOCATION` is the **agent's** location, normally `global`.
DATA_AGENT_PROJECT = "[your-data-agent-project]"  # @param {type:"string"}
DATA_AGENT_LOCATION = "global"  # @param {type:"string"}
DATA_AGENT_ID = "[your-data-agent-id]"  # @param {type:"string"}

# @markdown ### 1b. Where the evaluation runs — *Vertex AI (Agent Platform)* API
# @markdown `EVAL_PROJECT` holds the `EvaluationSet` + `EvaluationRun`. `REGION`
# @markdown is the **evaluation's** Vertex AI (Agent Platform) region and must be a real region,
# @markdown never `global`.
EVAL_PROJECT = "[your-eval-project]"  # @param {type:"string"}
REGION = "us-central1"  # @param {type:"string"}
# @markdown - **`SERVICE_ACCOUNT`**: a service account in `EVAL_PROJECT` that the
# @markdown   `EvaluationRun` uses to chat with the data agent, so the agent's BigQuery
# @markdown   SQL runs as this account. The metric only compares results; it does not
# @markdown   run SQL as you. When `USE_GCS_IMPORT=True`, also grant it access to
# @markdown   `STAGING_BUCKET`. Required for every data agent evaluation run
# @markdown   (format: `<name>@<EVAL_PROJECT>.iam.gserviceaccount.com`).
SERVICE_ACCOUNT = "[your-sa]@[your-eval-project].iam.gserviceaccount.com"  # @param {type:"string"}
# @markdown - **`INFERENCE_STALL_MIN`**: step 9 prints a warning when the run sits in
# @markdown   the same non-terminal state for this many minutes. The notebook does **not**
# @markdown   test table access as `SERVICE_ACCOUNT` (that would need you to hold
# @markdown   `roles/iam.serviceAccountTokenCreator`, which is not part of the customer
# @markdown   setup); the service checks it at run time. A table that
# @markdown   `SERVICE_ACCOUNT` cannot read can keep a run in INFERENCE for a long
# @markdown   time before it fails, so a stalled run is the signal to check the
# @markdown   prerequisite cell, steps 2-3.
INFERENCE_STALL_MIN = 20  # @param {type:"integer"}

# @markdown ---
# @markdown ### 2. Dataset Build Mode & Metric
# @markdown - **`cold`**: Synthesizes grounded golden queries from schema + real values.
# @markdown - **`seeded`**: Grounds synthetic generation on few-shot `(question, golden_sql)` examples in `SEED_PAIRS` defined below.
# @markdown - **`manual`**: Validates hand-written `MANUAL_PAIRS` defined below.
MODE = "cold"  # @param ["cold", "seeded", "manual"]
EVAL_SET_NAME = "data_agent_eval"  # @param {type:"string"}
NUM_PAIRS = 10  # @param {type:"integer"}
REPAIR_MODEL = "gemini-2.5-flash"  # @param {type:"string"}
MAX_REPAIRS = 3  # @param {type:"integer"}
MAX_GB_BILLED = 5  # @param {type:"number"}
# @markdown - **`KEEP_EMPTY_RESULTS`**: also register golden queries whose SQL is valid but
# @markdown   returns 0 rows — or 1 row of all `NULL`s from an aggregate over an unmatched
# @markdown   filter. Leave `False` to hold them as `NEEDS_REVIEW` and eyeball them first.
KEEP_EMPTY_RESULTS = False  # @param {type:"boolean"}
# @markdown - **`KEEP_DRY_RUN_ONLY`**: also register golden queries that passed the dry run but
# @markdown   were not executed: `TOO_BIG` (would scan more than `MAX_GB_BILLED`) or
# @markdown   `TIMEOUT` (slower than 20 s). These are never sent to LLM repair. Leave
# @markdown   `False` to hold them as `NEEDS_REVIEW`; set `True` for very large tables.
KEEP_DRY_RUN_ONLY = False  # @param {type:"boolean"}
METRIC = "bq_ca_data_result_accuracy_v1"  # @param {type:"string"}

# @markdown ---
# @markdown ### 3. Registration Method Switch (`inline_backoff` vs `gcs_import`)
# @markdown - **`USE_GCS_IMPORT = False` (`inline_backoff`)**: Creates items one by one through the API using 10 concurrent workers with automatic HTTP 429/5xx exponential backoff. **No Cloud Storage bucket needed.**
# @markdown - **`USE_GCS_IMPORT = True` (`gcs_import`)**: Uploads JSONL to GCS and calls `evaluationSets:import` + automatically configures `outputConfig.gcsDestination` during `create_run`. Recommended for large datasets.
USE_GCS_IMPORT = True  # @param {type:"boolean"}
# @markdown - **`STAGING_BUCKET`**: holds the import JSONL and the run outputs. It does
# @markdown   **not** have to live in `EVAL_PROJECT` or `DATA_AGENT_PROJECT`. When
# @markdown   `USE_GCS_IMPORT = True`, three identities need access: **you** (to upload
# @markdown   the JSONL), the **Vertex AI (Agent Platform) Service Agent** of `EVAL_PROJECT` (for
# @markdown   `evaluationSets:import`), and **`SERVICE_ACCOUNT`** (run outputs are
# @markdown   written to `outputConfig.gcsDestination` in the same bucket).
STAGING_BUCKET = "gs://[your-staging-bucket]"  # @param {type:"string"}

# MODE="manual": hand-written (question, golden_sql) pairs to validate and register directly.
_BQ = "`bigquery-public-data.austin_bikeshare.bikeshare_trips`"
MANUAL_PAIRS = [
    {
        "question": "How many trips were taken on electric bikes?",
        "golden_sql": (
            f"SELECT COUNT(*) AS trips FROM {_BQ} WHERE bike_type = 'electric'"
        ),
    },
]
# MODE="seeded": few-shot (question, golden_sql) examples passed to DataFoundry
# so synthetic generation follows your SQL style, joins, and question phrasing.
SEED_PAIRS = [
    {
        "question": "How many trips were taken on electric bikes?",
        "golden_sql": (
            f"SELECT COUNT(*) AS trips FROM {_BQ} WHERE bike_type = 'electric'"
        ),
    },
]

# ---- Derived (do not edit) ----

DATA_AGENT_PROJECT = DATA_AGENT_PROJECT.strip()
DATA_AGENT_LOCATION = DATA_AGENT_LOCATION.strip()
DATA_AGENT_ID = DATA_AGENT_ID.strip()
EVAL_PROJECT = EVAL_PROJECT.strip()
REGION = REGION.strip()
SERVICE_ACCOUNT = SERVICE_ACCOUNT.strip()
STAGING_BUCKET = STAGING_BUCKET.strip()

# Fail fast if any required form field still holds its placeholder value.
_REQUIRED_FIELDS = {
    "DATA_AGENT_PROJECT": DATA_AGENT_PROJECT,
    "DATA_AGENT_ID": DATA_AGENT_ID,
    "EVAL_PROJECT": EVAL_PROJECT,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
}
if USE_GCS_IMPORT:
  _REQUIRED_FIELDS["STAGING_BUCKET"] = STAGING_BUCKET
_UNEDITED = [
    name
    for name, val in _REQUIRED_FIELDS.items()
    if not val
    or any(ch in val for ch in "[]<>")
    or re.search(r"(^|/|@)your-", val, re.IGNORECASE)
]
assert not _UNEDITED, (
    "Edit the form fields in the Configuration cell before running: "
    f"{', '.join(_UNEDITED)} still contain unedited placeholder values."
)

# Validate DATA_AGENT_LOCATION and resolve the geminidataanalytics API endpoint.
# Per geminidataanalytics service configuration:
# - Multi-regions ("us", "eu") and "europe-west4" use geminidataanalytics.{loc}.rep.googleapis.com
# - Other supported single regions use geminidataanalytics-{loc}.googleapis.com
_AGENT_REP_LOCATIONS = frozenset({"us", "eu", "europe-west4"})
_AGENT_SINGLE_REGIONS = frozenset({
    "asia-northeast1",
    "europe-west2",
    "northamerica-northeast1",
    "northamerica-northeast2",
    "us-east4",
    "us-east7",
    "us-south1",
    "us-west1",
})


def data_agent_api_endpoint(location):
  """Returns the geminidataanalytics API hostname for a supported location."""
  if location == "global":
    return "geminidataanalytics.googleapis.com"
  if location in _AGENT_REP_LOCATIONS:
    return f"geminidataanalytics.{location}.rep.googleapis.com"
  if location in _AGENT_SINGLE_REGIONS:
    return f"geminidataanalytics-{location}.googleapis.com"
  supported = sorted({"global"} | _AGENT_REP_LOCATIONS | _AGENT_SINGLE_REGIONS)
  raise ValueError(
      f"Unsupported DATA_AGENT_LOCATION={location!r}. Supported locations: "
      f"{', '.join(supported)}. See "
      "https://cloud.google.com/gemini/docs/conversational-analytics-api/data-residency"
  )


DATA_AGENT_ENDPOINT = data_agent_api_endpoint(DATA_AGENT_LOCATION)
API_VERSION = "v1beta1"
ENDPOINT = f"{REGION}-aiplatform.googleapis.com"
CANDIDATE = "data_agent"
POLL_INTERVAL_S = 15
TIMEOUT_S = 3600 * 5
RETRYABLE = (429, 500, 502, 503, 504)
DATA_AGENT_RESOURCE = (
    f"projects/{DATA_AGENT_PROJECT}/locations/{DATA_AGENT_LOCATION}"
    f"/dataAgents/{DATA_AGENT_ID}"
)
BARE_BUCKET = STAGING_BUCKET.replace("gs://", "").strip("/")
assert MODE in ("cold", "seeded", "manual"), f"Invalid MODE: {MODE}"
assert REGION != "global", (
    "REGION is the Vertex AI (Agent Platform) *evaluation* region and must be a"
    " real region such as us-central1. If you meant the data agent's location,"
    " set DATA_AGENT_LOCATION='global' instead -- they are different knobs."
)
assert (
    SERVICE_ACCOUNT
), "SERVICE_ACCOUNT is required for data agent evaluation runs."

creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
try:
  creds = creds.with_quota_project(EVAL_PROJECT)
except AttributeError:
  pass


def strip_fences(sql):
  """Strips markdown code fences (e.g. ```sql, ```GoogleSQL, ```bigquery)."""
  sql = re.sub(
      r"^```[a-zA-Z0-9_-]*\s*", "", (sql or "").strip(), flags=re.IGNORECASE
  )
  return re.sub(r"\s*```$", "", sql).strip()


def _err_line(e, limit=200):
  """Returns the first non-empty line of an exception message, else its type name."""
  for line in str(e).splitlines():
    if line.strip():
      return line.strip()[:limit]
  return type(e).__name__


# EVAL_PROJECT's Vertex AI (Agent Platform) Service Agent domain; the project
# number is filled in by the cells that print IAM hints (the prerequisite cell
# shows how to look it up).
SERVICE_AGENT_DOMAIN = "gcp-sa-aiplatform.iam.gserviceaccount.com"


def _iam_hint(text):
  """Maps a known 403 text to the prerequisite cell step that fixes it, else ""."""
  t = str(text)
  if "serviceusage.services.use" in t:
    return (
        f"You need roles/serviceusage.serviceUsageConsumer on {EVAL_PROJECT}"
        " (prerequisite cell, step 0): Vertex AI (Agent Platform) API calls"
        " from this notebook use EVAL_PROJECT as the quota project via"
        " x-goog-user-project, and roles/aiplatform.user does not include"
        " serviceusage.services.use."
    )
  if "iam.serviceAccounts.actAs" in t:
    return (
        f"You need roles/iam.serviceAccountUser on {SERVICE_ACCOUNT} "
        "(prerequisite cell, step 1)."
    )
  return ""


class ApiError(RuntimeError):
  """A non-2xx response; .status tells a permanent 4xx from a transient error."""

  def __init__(self, method, path, status, text):
    hint = _iam_hint(text)
    super().__init__(
        f"{method} {path} -> {status}: {text[:300]}"
        + (f"\n  -> {hint}" if hint else "")
    )
    self.status = status


# Shared AuthorizedSession and retry helpers used across Section A and Section B.
_eval_session = AuthorizedSession(creds)
_eval_session.headers.update({
    "Content-Type": "application/json",
    "x-goog-user-project": EVAL_PROJECT,
})


def _base():
  return f"https://{ENDPOINT}/{API_VERSION}"


def rest_post(path, body, attempts=6, timeout=120):
  """POSTs to Vertex AI (Agent Platform) evaluation API with retry on 429/5xx and network errors."""
  url = path if path.startswith("https://") else f"{_base()}/{path}"
  last = None
  for attempt in range(attempts):
    try:
      r = _eval_session.post(url, json=body, timeout=timeout)
    except requests.exceptions.RequestException as e:
      last = f"{type(e).__name__}: {_err_line(e, 200)}"
    else:
      if r.ok:
        return r.json()
      if r.status_code not in RETRYABLE:
        raise ApiError("POST", path, r.status_code, r.text)
      last = f"status={r.status_code} {r.text[:200]}"
    if attempt < attempts - 1:
      time.sleep(2**attempt + random.uniform(0.1, 1.0))
  raise RuntimeError(
      f"POST {path} failed after {attempts} attempts (last: {last})"
  )


def rest_get(path, attempts=5, timeout=60):
  """GETs from Vertex AI (Agent Platform) evaluation API with retry on 429/5xx and network errors."""
  url = path if path.startswith("https://") else f"{_base()}/{path}"
  last = None
  for attempt in range(attempts):
    try:
      r = _eval_session.get(url, timeout=timeout)
    except requests.exceptions.RequestException as e:
      last = f"{type(e).__name__}: {_err_line(e, 200)}"
    else:
      if r.ok:
        return r.json()
      if r.status_code not in RETRYABLE:
        raise ApiError("GET", path, r.status_code, r.text)
      last = f"status={r.status_code} {r.text[:200]}"
    if attempt < attempts - 1:
      time.sleep(2**attempt + random.uniform(0.1, 1.0))
  raise RuntimeError(
      f"GET {path} failed after {attempts} attempts (last: {last})"
  )


bq = bigquery.Client(project=DATA_AGENT_PROJECT, credentials=creds)
MAX_BYTES = int(MAX_GB_BILLED * 1e9)
# Keep pairs from Section A if this cell is re-run (e.g. to fix REGION or
# SERVICE_ACCOUNT before registering). Step 4 still starts a fresh list.
pairs = globals().get("pairs", [])

print(f"MODE : {MODE}")
print("-- agent under test (geminidataanalytics) " + "-" * 32)
print(f"   data agent     : {DATA_AGENT_RESOURCE}")
print(f"   agent location : {DATA_AGENT_LOCATION} ({DATA_AGENT_ENDPOINT})")
print(f"   bigquery       : {DATA_AGENT_PROJECT}")
print("-- evaluation (Vertex AI / Agent Platform) " + "-" * 31)
print(f"   eval project   : {EVAL_PROJECT}")
print(f"   eval region    : {REGION}")
print(f"   endpoint       : {ENDPOINT}")
print(f"   service account: {SERVICE_ACCOUNT}")
print("-- registration " + "-" * 59)
print(
    f"   USE_GCS_IMPORT : {USE_GCS_IMPORT}"
    f" ({'gcs_import' if USE_GCS_IMPORT else 'inline_backoff'})"
)
if USE_GCS_IMPORT:
  print(
      f"   staging bucket : gs://{BARE_BUCKET}  (GCS quota project:"
      f" {EVAL_PROJECT})"
  )

## Prerequisite — service account and IAM grants

Every data agent evaluation run needs a **customer-managed service account**,
passed as `SERVICE_ACCOUNT` (`EvaluationRun.service_account`). The
Vertex AI (Agent Platform) Service Agent of `EVAL_PROJECT`
(`service-<EVAL_PROJECT_NUMBER>@gcp-sa-aiplatform.iam.gserviceaccount.com`)
impersonates `SERVICE_ACCOUNT` to chat with the data agent, and the agent runs its
BigQuery SQL as `SERVICE_ACCOUNT`. The metric compares results; it does not run SQL
as you.

**What you (the notebook user) need:**

| Where | Role | Why |
|---|---|---|
| `EVAL_PROJECT` | `roles/aiplatform.user` | create the `EvaluationSet` / `EvaluationRun`, read results |
| `EVAL_PROJECT` | `roles/serviceusage.serviceUsageConsumer` (step 0) | sets `EVAL_PROJECT` as the quota project for Vertex AI (Agent Platform) evaluation API calls (`serviceusage.services.use`, which `roles/aiplatform.user` does not include; BigQuery jobs inside the agent are billed to the BigQuery project such as `DATA_AGENT_PROJECT`) |
| `SERVICE_ACCOUNT` | `roles/iam.serviceAccountUser` (step 1) | attach the service account to the run (`iam.serviceAccounts.actAs`). **`SERVICE_ACCOUNT` must belong to `EVAL_PROJECT`** |
| `DATA_AGENT_PROJECT` | `roles/geminidataanalytics.dataAgentViewer`, `roles/bigquery.jobUser`, `roles/bigquery.dataViewer` on the agent's tables | read the agent definition; ground and validate the golden queries **as you** |
| `STAGING_BUCKET` (when `USE_GCS_IMPORT=True`) | `roles/storage.objectUser` | upload the dataset file; read run outputs |

> **You do not need `roles/iam.serviceAccountTokenCreator` on `SERVICE_ACCOUNT`.**
> This notebook never impersonates `SERVICE_ACCOUNT`; only the
> Vertex AI (Agent Platform) Service Agent does (step 2). The service checks that
> `SERVICE_ACCOUNT` can read the agent's tables while the run executes. Step 9
> warns if a run stalls (`INFERENCE_STALL_MIN`).

Run these grants once before your first evaluation run:

```bash
PN=$(gcloud projects describe EVAL_PROJECT --format="value(projectNumber)")
SERVICE_AGENT="service-${PN}@gcp-sa-aiplatform.iam.gserviceaccount.com"
SA="SERVICE_ACCOUNT"
CALLER_EMAIL=$(gcloud config get-value account)

# 0. Let the caller (you) use EVAL_PROJECT as the quota project for Vertex AI (Agent Platform) evaluation API calls
#    (BigQuery queries executed by the agent are billed to DATA_AGENT_PROJECT).
gcloud projects add-iam-policy-binding EVAL_PROJECT \
  --member="user:${CALLER_EMAIL}" \
  --role="roles/serviceusage.serviceUsageConsumer"

# 1. Create the service account in EVAL_PROJECT (skip if it exists) and let the
#    caller (you) attach it to a run. A service account from another project is
#    rejected by the default org policy iam.disableCrossProjectServiceAccountUsage.
# gcloud iam service-accounts create data-agent-eval-sa --project=EVAL_PROJECT
gcloud iam service-accounts add-iam-policy-binding "${SA}" \
  --member="user:${CALLER_EMAIL}" \
  --role="roles/iam.serviceAccountUser"

# 2. Let the Vertex AI (Agent Platform) Service Agent of EVAL_PROJECT impersonate the service account.
gcloud iam service-accounts add-iam-policy-binding "${SA}" \
  --member="serviceAccount:${SERVICE_AGENT}" \
  --role="roles/iam.serviceAccountTokenCreator"

# 3. Grant the service account what it needs in DATA_AGENT_PROJECT.
# 3a. Project-wide: call the conversation API and run BigQuery jobs.
for R in roles/cloudaicompanion.user roles/bigquery.jobUser; do
  gcloud projects add-iam-policy-binding DATA_AGENT_PROJECT \
    --member="serviceAccount:${SA}" --role="$R"
done
# 3b. dataAgentUser on the one agent under evaluation, through the REST IAM methods.
#     Match the endpoint hostname to DATA_AGENT_LOCATION:
AGENT="projects/DATA_AGENT_PROJECT/locations/DATA_AGENT_LOCATION/dataAgents/DATA_AGENT_ID"
case "DATA_AGENT_LOCATION" in
  global)
    API_HOST="https://geminidataanalytics.googleapis.com/v1beta"
    ;;
  us|eu|europe-west4)
    API_HOST="https://geminidataanalytics.DATA_AGENT_LOCATION.rep.googleapis.com/v1beta"
    ;;
  asia-northeast1|europe-west2|northamerica-northeast1|northamerica-northeast2|us-east4|us-east7|us-south1|us-west1)
    API_HOST="https://geminidataanalytics-DATA_AGENT_LOCATION.googleapis.com/v1beta"
    ;;
  *)
    echo "Unsupported DATA_AGENT_LOCATION: DATA_AGENT_LOCATION (see https://cloud.google.com/gemini/docs/conversational-analytics-api/data-residency)" >&2
    exit 1
    ;;
esac
TOKEN=$(gcloud auth print-access-token)
curl -fsS -X POST -H "Authorization: Bearer ${TOKEN}" -H "Content-Type: application/json" \
  -d '{}' "${API_HOST}/${AGENT}:getIamPolicy" > /tmp/agent_policy.json && \
jq -e --arg m "serviceAccount:${SA}" \
  '.bindings = ((.bindings // []) + [{"role":"roles/geminidataanalytics.dataAgentUser","members":[$m]}])' \
  /tmp/agent_policy.json > /tmp/agent_policy_new.json && \
curl -fsS -X POST -H "Authorization: Bearer ${TOKEN}" -H "Content-Type: application/json" \
  -d "{\"policy\": $(cat /tmp/agent_policy_new.json)}" "${API_HOST}/${AGENT}:setIamPolicy"
# 3c. dataViewer on the agent's datasets (repeat for each dataset listed in step 1):
bq query --use_legacy_sql=false --project_id=DATA_AGENT_PROJECT \
  "GRANT \`roles/bigquery.dataViewer\` ON SCHEMA \`DATA_AGENT_PROJECT.DATASET\` TO \"serviceAccount:${SA}\""
# Project-wide roles/geminidataanalytics.dataAgentUser and roles/bigquery.dataViewer
# on DATA_AGENT_PROJECT also work; they are just wider than needed.

# 4. When USE_GCS_IMPORT=True: grant bucket access to the service account
#    (run creation checks storage.buckets.get; outputs are written to the bucket)
#    and to the Vertex AI (Agent Platform) Service Agent (for evaluationSets:import).
for R in roles/storage.objectAdmin roles/storage.legacyBucketReader; do
  gcloud storage buckets add-iam-policy-binding gs://BARE_BUCKET \
    --member="serviceAccount:${SA}" --role="$R"
done
gcloud storage buckets add-iam-policy-binding gs://BARE_BUCKET \
  --member="serviceAccount:${SERVICE_AGENT}" \
  --role="roles/storage.objectAdmin"
```

Replace `EVAL_PROJECT`, `SERVICE_ACCOUNT`, `DATA_AGENT_PROJECT`, `DATA_AGENT_LOCATION`, `DATA_AGENT_ID`, `DATASET`, and `BARE_BUCKET`. Wait a couple of minutes for IAM changes to propagate.

**Troubleshooting**
- `403 ... serviceusage.services.use`: step 0 is missing.
- `CreateEvaluationRun` fails with `400 INVALID_ARGUMENT` or `403 PERMISSION_DENIED`: check that `SERVICE_ACCOUNT` is set, exists, **belongs to `EVAL_PROJECT`**, that you have `roles/iam.serviceAccountUser` on it, and that it has step 4's roles on `STAGING_BUCKET`.
- Per-question `PERMISSION_DENIED`: `User does not have permission to chat` means step 3b; `bigquery.jobs.create` means step 3a; `User does not have permission to query table` means step 3c; a token or `getAccessToken` error means step 2.
- To remove a stuck or unwanted evaluation run, call `delete_run(run_name)` in step 8. **Warning:** `delete_run()` permanently deletes the `EvaluationRun` resource and its evaluation results.

# Section A — Build the evaluation dataset

Pick `MODE` in the config cell: `cold` synthesizes grounded golden queries from the
schema + real values; `seeded` grounds synthetic generation on your few-shot
`SEED_PAIRS`; `manual` validates your own `MANUAL_PAIRS`. This section reads
the agent's schema, generates/validates the golden queries, lets you review them, and
registers the `EvaluationSet` under `EVAL_PROJECT`. It sets `EVAL_SET`, which
Section B uses automatically.

## 1. Fetch agent scope, schema, **and real values**  ·  `cold` `seeded`

Read the agent's BigQuery tables + schemas, and pull the **actual distinct
values** for low-cardinality columns and the **date range** of each time column.
This is what stops the generator inventing literals (e.g. `bike_type =
'electric_bike'` when the real value is `electric`) — the #1 cause of
valid-but-empty golden SQL.

The grounding scan is **capped at `MAX_GB_BILLED`** and is best-effort: if a
table requires a partition filter, the scan adds `IS NOT NULL` on the partition
column; if the scan would still exceed the cap, grounding is **skipped for that
table** (no crash, no runaway bill). `ARRAY` and `STRUCT` columns are shown with
their full type and are not value-grounded. Only columns with ≤10 distinct
values become `[allowed values: …]` hints; higher-cardinality columns are left
unconstrained.

Step 1 also confirms that `SERVICE_ACCOUNT` **exists** (a typo here otherwise
surfaces only at `CreateEvaluationRun`). It does **not** test table access as
`SERVICE_ACCOUNT`: that would require you to hold
`roles/iam.serviceAccountTokenCreator` on it, which is not part of the customer
setup. Keep in mind that golden queries are grounded and validated **as you**, while
the agent under evaluation queries **as `SERVICE_ACCOUNT`** -- if the two see
different rows (Row-Level Security, policy tags), scores will be off.

In [ ]:
# @title 1. Scope + schema + value grounding
tables, system_instruction, schema_text = [], "", ""
ID_LIKE = re.compile(r"(^|_)id$", re.IGNORECASE)


def _col_type(f):
  """Type text for a schema field, e.g. ARRAY<STRING> or STRUCT<city STRING>."""
  t = f.field_type
  if t in ("RECORD", "STRUCT"):
    t = (
        "STRUCT<"
        + ", ".join(f"{s.name} {_col_type(s)}" for s in f.fields)
        + ">"
    )
  return f"ARRAY<{t}>" if f.mode == "REPEATED" else t


def _partition_filter(meta):
  """A WHERE clause that satisfies require_partition_filter, else ""."""
  if not getattr(meta, "require_partition_filter", False):
    return ""
  tp = getattr(meta, "time_partitioning", None)
  rp = getattr(meta, "range_partitioning", None)
  if tp is not None:
    # Ingestion-time partitioning has no column; filter the pseudo-column.
    return (
        f" WHERE `{tp.field}` IS NOT NULL"
        if tp.field
        else " WHERE _PARTITIONTIME IS NOT NULL"
    )
  if rp is not None and getattr(rp, "field", None):
    return f" WHERE `{rp.field}` IS NOT NULL"
  return ""


# DATA_AGENT_LOCATION, not REGION: the data agent is a geminidataanalytics
# resource with its own location namespace and its own hostname (resolved by
# data_agent_api_endpoint() in the Configuration cell).
agent_kwargs = {"credentials": creds}
if DATA_AGENT_LOCATION != "global":
  _agent_host = data_agent_api_endpoint(DATA_AGENT_LOCATION)
  agent_kwargs["client_options"] = client_options_lib.ClientOptions(
      api_endpoint=_agent_host
  )
  print(f"Regional data agent: using {_agent_host}")
agent_client = geminidataanalytics.DataAgentServiceClient(**agent_kwargs)
agent = agent_client.get_data_agent(name=DATA_AGENT_RESOURCE)
da = getattr(agent, "data_analytics_agent", None)
ctx, ctx_kind = None, ""
if da:
  pub = getattr(da, "published_context", None)
  staging_ctx = getattr(da, "staging_context", None)
  if pub and (pub.system_instruction or pub.datasource_references):
    ctx, ctx_kind = pub, "published_context"
  elif staging_ctx and (
      staging_ctx.system_instruction or staging_ctx.datasource_references
  ):
    ctx, ctx_kind = staging_ctx, "staging_context"
system_instruction = ctx.system_instruction if ctx else ""
if ctx_kind == "staging_context":
  print(
      "WARNING: this agent has no published_context; golden queries will be"
      " built from its staging_context. The EvaluationRun queries the agent's"
      " PUBLISHED context, so publish the agent before running step 9 or the"
      " run will not see these tables."
  )
elif ctx_kind:
  print(f"Agent context used for schema + golden queries: {ctx_kind}")

if ctx and ctx.datasource_references and ctx.datasource_references.bq:
  for t in ctx.datasource_references.bq.table_references:
    fq = f"{t.project_id or DATA_AGENT_PROJECT}.{t.dataset_id}.{t.table_id}"
    meta = bq.get_table(fq)
    cols = [
        {"name": c.name, "type": _col_type(c), "desc": c.description or ""}
        for c in meta.schema
    ]
    grounding = {}
    if MODE in ("cold", "seeded"):
      sel, str_cols, time_cols = [], [], []
      for c in cols:
        if c["type"] == "STRING" and not ID_LIKE.search(c["name"]):
          str_cols.append(c["name"])
        elif c["type"] in ("TIMESTAMP", "DATE", "DATETIME"):
          time_cols.append(c["name"])
      if len(str_cols) > 50:
        print(
            f"  note: {fq} has {len(str_cols)} STRING columns; grounding the"
            " first 50."
        )
      str_cols = str_cols[:50]
      for i, c in enumerate(str_cols):
        sel.append(f"APPROX_TOP_COUNT(`{c}`, 11) AS top_{i}")
      for i, c in enumerate(time_cols):
        sel.append(f"MIN(`{c}`) AS min_{i}")
        sel.append(f"MAX(`{c}`) AS max_{i}")
      if sel:
        where_clause = _partition_filter(meta)
        job_cfg = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
        try:
          row = list(
              bq.query(
                  f"SELECT {', '.join(sel)} FROM `{fq}`{where_clause}",
                  job_config=job_cfg,
              ).result()
          )[0]
          for i, c in enumerate(str_cols):
            vals = [
                v["value"] for v in row[f"top_{i}"] if v["value"] is not None
            ]
            if 0 < len(vals) <= 10:
              grounding[c] = vals
          for i, c in enumerate(time_cols):
            if row[f"min_{i}"] is not None and row[f"max_{i}"] is not None:
              grounding[c] = f"{row[f'min_{i}']} .. {row[f'max_{i}']}"
        except Exception as e:  # pylint: disable=broad-except
          print(f"  grounding skipped for {fq}: {_err_line(e, 120)}")
    # Views and external tables report num_rows=0; say so instead of "0 rows".
    _kind = (getattr(meta, "table_type", "") or "").upper()
    _rows = (
        _kind.lower()
        if _kind in ("VIEW", "EXTERNAL", "MATERIALIZED_VIEW")
        else meta.num_rows
    )
    tables.append(
        {"fq": fq, "rows": _rows, "columns": cols, "grounding": grounding}
    )

if MODE in ("cold", "seeded"):
  blocks = []
  for t in tables:
    lines = []
    for c in t["columns"]:
      g = t["grounding"].get(c["name"])
      hint = ""
      if isinstance(g, list):
        hint = f"  [allowed values: {', '.join(repr(x) for x in g)}]"
      elif isinstance(g, str):
        hint = f"  [range: {g}]"
      lines.append(
          f"  - `{c['name']}` ({c['type']}): "
          f"{c['desc'] or 'No description'}{hint}"
      )
    blocks.append(
        f"Table: `{t['fq']}` ({t['rows']} rows)\nColumns:\n" + "\n".join(lines)
    )
  schema_text = "\n\n".join(blocks)

for t in tables:
  print(f"Agent Table: {t['fq']} ({t['rows']} rows)")
  if MODE in ("cold", "seeded"):
    for c, v in t["grounding"].items():
      print(f"  {c}: {v}")
if MODE not in ("cold", "seeded"):
  print(f"MODE={MODE}: value grounding scan skipped")


def _check_service_account_exists():
  """Fails fast on a misspelled or disabled SERVICE_ACCOUNT.

  Only calls GET on the service account (iam.serviceAccounts.get, which
  roles/iam.serviceAccountUser grants). It never mints a token for the account:
  that would require roles/iam.serviceAccountTokenCreator, which is not part
  of the customer setup. Table access as SERVICE_ACCOUNT is checked by the
  service when the EvaluationRun executes.
  """
  # SERVICE_ACCOUNT must belong to EVAL_PROJECT: the default org policy
  # iam.disableCrossProjectServiceAccountUsage stops Vertex AI (Agent Platform)
  # from attaching another project's service account, and the backend reports
  # that only as a generic 403 PERMISSION_DENIED at CreateEvaluationRun.
  m = re.fullmatch(
      r"[^@]+@([a-z0-9-]+)\.iam\.gserviceaccount\.com", SERVICE_ACCOUNT
  )
  if m and m.group(1) != EVAL_PROJECT:
    raise AssertionError(
        f"SERVICE_ACCOUNT {SERVICE_ACCOUNT} belongs to project {m.group(1)!r},"
        f" but EVAL_PROJECT is {EVAL_PROJECT!r}. Create the service account"
        f" inside {EVAL_PROJECT} (prerequisite cell, step 1) and grant it"
        f" access in {DATA_AGENT_PROJECT} (prerequisite cell, step 3)."
        " Cross-project service accounts are blocked by the org policy"
        " iam.disableCrossProjectServiceAccountUsage."
    )
  url = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
  try:
    r = AuthorizedSession(creds).get(url, timeout=15)
  except Exception as e:  # pylint: disable=broad-except
    print(
        f"NOTE: could not verify that {SERVICE_ACCOUNT} exists"
        f" ({_err_line(e, 120)})."
    )
    return
  if r.status_code == 404:
    raise AssertionError(
        f"SERVICE_ACCOUNT {SERVICE_ACCOUNT} does not exist (HTTP 404). Check"
        " the email for typos: <name>@<project>.iam.gserviceaccount.com."
    )
  if r.status_code == 403:
    # Without iam.serviceAccounts.get the caller also lacks actAs, so
    # CreateEvaluationRun would fail later with the same missing grant.
    raise AssertionError(
        f"No permission to look up {SERVICE_ACCOUNT} (HTTP 403; this can also"
        " mean the account does not exist). You need"
        " roles/iam.serviceAccountUser on it (prerequisite cell, step 1) to"
        " attach it to a run. "
        + (_iam_hint(r.text) or "")
    )
  if not r.ok:
    print(
        f"NOTE: SERVICE_ACCOUNT lookup returned HTTP {r.status_code};"
        " continuing."
    )
    return
  if r.json().get("disabled"):
    raise AssertionError(
        f"SERVICE_ACCOUNT {SERVICE_ACCOUNT} is disabled; enable it before"
        " starting a run."
    )
  print(f"SERVICE_ACCOUNT exists: {SERVICE_ACCOUNT}")


_check_service_account_exists()

# Fail loudly here: with an empty scope, schema_text is "" and step 4 generates
# plausible-looking pairs against tables that do not exist. That only surfaces
# much later as a wall of unexplained validation errors.
assert tables or MODE not in ("cold", "seeded"), (
    "Agent resolved to 0 BigQuery tables, so schema_text is empty and"
    " generation would invent tables. Check DATA_AGENT_ID, and that the agent"
    " has a published (or staging) context with datasource_references.bq set."
)

## 2. Prepare few-shot seeds  ·  `seeded`

In **`seeded`** mode, DataFoundry takes the example `(question, golden_sql)`
pairs from `SEED_PAIRS` in the Configuration cell as few-shot examples when
generating `NUM_PAIRS` new synthetic pairs. Edit `SEED_PAIRS` in Cell 3 (or
override `SEEDS` below) with 1–5 representative queries for your agent.


In [ ]:
# @title 2. Prepare few-shot seeds (seeded)
SEEDS = []
if MODE == "seeded":
  assert SEED_PAIRS, (
      "MODE='seeded' requires at least one example in SEED_PAIRS (see the "
      "Configuration cell), or switch to MODE='cold'."
  )
  SEEDS = [
      {
          "question": s["question"].strip(),
          "golden_sql": strip_fences(s["golden_sql"]),
      }
      for s in SEED_PAIRS
      if s.get("question") and s.get("golden_sql")
  ]
  assert (
      SEEDS
  ), "SEED_PAIRS entries must each have non-empty 'question' and 'golden_sql'."
  print(f"seeded: using {len(SEEDS)} few-shot seed(s) from SEED_PAIRS")
  for i, s in enumerate(SEEDS):
    print(f"  [{i+1}] Q  : {s['question'][:80]}")
    print(f"      SQL: {' '.join(s['golden_sql'].split())[:90]}")
else:
  print(f"MODE={MODE}: no few-shot seeds needed -> skipped")

## 3. Build the generation task  ·  `cold` `seeded`

In [ ]:
# @title 3. TaskDescriptionStrategy + OutputFieldSpec
task_prompt, specs = "", []
if MODE in ("cold", "seeded"):
  task_prompt = f"""You are generating a single-turn evaluation dataset for a \
BigQuery Conversational Analytics agent. The agent answers questions by \
generating and running GoogleSQL.

AGENT ROLE & INSTRUCTIONS:
{system_instruction or 'Help users query and analyze BigQuery data accurately.'}

AVAILABLE TABLES, SCHEMAS, AND REAL VALUES:
{schema_text}

REQUIREMENTS:
1. Generate realistic, diverse analytical questions (filtering, time-series
   aggregation, top-N ranking, grouping, calculations).
2. For each question generate the golden_sql in valid GoogleSQL.
3. Read-only SELECT only. Reference exact `project.dataset.table` in backticks.
4. Use ONLY column names from the schema, and when filtering a column that lists
   allowed values, use ONLY those exact values and stay within listed ranges.
   Most questions should return data; a few intentional empty-result (boundary)
   questions are fine and welcome."""
  specs = [
      aiplatform_v1.OutputFieldSpec(
          field_name="question",
          guidance="A realistic, concise NL question a user would ask.",
          field_type=aiplatform_v1.OutputFieldSpec.FieldType.TEXT,
      ),
      aiplatform_v1.OutputFieldSpec(
          field_name="golden_sql",
          guidance="A valid GoogleSQL query answering the question.",
          field_type=aiplatform_v1.OutputFieldSpec.FieldType.TEXT,
      ),
  ]
  print(task_prompt[:1000], "\n...[truncated]...")
else:
  print(f"MODE={MODE}: generation task not needed -> skipped")

## 4. Generate synthetic pairs (DataFoundryService)  ·  `cold` `seeded`

In [ ]:
# @title 4. GenerateSyntheticData
if MODE in ("cold", "seeded"):
  # DataFoundry (synthetic data generation) runs on the regional
  # Vertex AI (Agent Platform) endpoint for REGION.
  df_client = aiplatform_v1.DataFoundryServiceClient(
      credentials=creds,
      client_options=client_options_lib.ClientOptions(
          api_endpoint=f"{REGION}-aiplatform.googleapis.com"
      ),
  )
  examples = []
  if MODE == "seeded":
    for s in SEEDS:
      examples.append(
          aiplatform_v1.SyntheticExample(
              fields=[
                  aiplatform_v1.SyntheticField(
                      field_name="question",
                      content={"parts": [{"text": s["question"]}]},
                  ),
                  aiplatform_v1.SyntheticField(
                      field_name="golden_sql",
                      content={"parts": [{"text": s["golden_sql"]}]},
                  ),
              ]
          )
      )
  # DataFoundry caps count at 1..50 per GenerateSyntheticData call.
  t0 = time.time()
  raw_examples = []
  while len(raw_examples) < NUM_PAIRS:
    batch_n = min(50, NUM_PAIRS - len(raw_examples))
    req = aiplatform_v1.GenerateSyntheticDataRequest(
        location=f"projects/{EVAL_PROJECT}/locations/{REGION}",
        count=batch_n,
        task_description=aiplatform_v1.TaskDescriptionStrategy(
            task_description=task_prompt
        ),
        output_field_specs=specs,
        examples=examples,
    )
    resp = df_client.generate_synthetic_data(request=req)
    if not resp.synthetic_examples:
      break
    raw_examples.extend(resp.synthetic_examples)
  print(
      f"Generated {len(raw_examples)} pairs in "
      f"{time.time()-t0:.1f}s (seeds={len(examples)})"
  )
  pairs = []
  for i, ex in enumerate(raw_examples):
    d = {}
    for f in ex.fields:
      d[f.field_name] = " ".join(
          p.text for p in f.content.parts if getattr(p, "text", None)
      ).strip()
    pairs.append({
        "index": i + 1,
        "question": d.get("question", "").strip(),
        "golden_sql": strip_fences(d.get("golden_sql", "")),
    })
  for p in pairs:
    print(f"[{p['index']}] {p['question']}")
elif MODE == "manual":
  assert MANUAL_PAIRS, "manual mode needs MANUAL_PAIRS in the config cell"
  pairs = [
      {
          "index": i + 1,
          "question": p["question"],
          "golden_sql": strip_fences(p["golden_sql"]),
      }
      for i, p in enumerate(MANUAL_PAIRS)
  ]
  print(f"manual: {len(pairs)} hand-written pair(s) to validate")
else:
  print(f"MODE={MODE}: generation skipped")

## 5. Validate safely on live BigQuery + repair  ·  `cold` `seeded` `manual`

Safe validation, in order:
1. **Dry run first** (`$0`) — get the statement type and syntax **without
   executing**. Reject anything that is not a `SELECT` *before* any job runs, so
   a hallucinated `DROP`/`DELETE`/`UPDATE` can never execute against your data.
2. **Estimate check** — if the dry run says the query would scan more than
   `MAX_GB_BILLED`, flag it (`NEEDS_REVIEW`) and do **not** run it.
3. **Bounded execution** — run with `maximum_bytes_billed` so a runaway
   cross-join fails fast instead of billing a huge scan.

Result classes: `ERROR`/`non-SELECT` → repair loop. **`0 rows` → flag only**, no
repair — an empty result is often the correct answer, so we never mutate a query
just to force data. `too big` → flag, never run. `TIMEOUT` (slower than 20 s) →
flag, never repaired: slow is not wrong. Set **`KEEP_DRY_RUN_ONLY = True`** to
register `too big` / `TIMEOUT` golden queries on the strength of the dry run alone.

> **`0 rows` (and `1 row of all NULLs`) are held back from registration by default.**
> Step 3 asks the generator for a few intentional boundary questions, and an
> aggregate without `GROUP BY` over an unmatched filter returns 1 row of `NULL`s,
> but step 7 only registers `VALID` pairs. Set **`KEEP_EMPTY_RESULTS = True`** in
> the config cell to promote `EMPTY` → `VALID` and keep them.

> In **`manual`** mode we **validate only** — failures are flagged
> `NEEDS_REVIEW` for you to fix. We never auto-repair a golden you wrote by hand.

In [ ]:
# @title 5. Safe validate (dry-run -> cap -> execute) and repair (Parallelized)
if MODE in ("cold", "seeded", "manual"):
  # Initialize the Gen AI client on the regional Vertex AI (Agent Platform)
  # endpoint for SQL repair (skipped in manual mode, which validates without
  # auto-repair).
  genai_client = None
  if MODE != "manual":
    genai_client = genai.Client(
        vertexai=True, project=EVAL_PROJECT, location=REGION, credentials=creds
    )

  def _cancel(job):
    if job is not None:
      try:
        job.cancel()
      except Exception:  # pylint: disable=broad-except
        pass

  def validate(sql):
    sql = strip_fences(sql)
    # 1. Dry run: statement type + syntax, no execution, no cost.
    try:
      dj = bq.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True))
    except Exception as e:  # pylint: disable=broad-except
      return "ERROR", _err_line(e, 200), 0
    if (dj.statement_type or "SELECT") != "SELECT":
      return "NON_SELECT", f"non-SELECT: {dj.statement_type}", 0
    est = dj.total_bytes_processed or 0
    if est > MAX_BYTES:
      return (
          "TOO_BIG",
          f"would scan {est/1e9:.3g} GB > {MAX_GB_BILLED} GB cap",
          0,
      )

    # 2. Bounded execution with timeout.
    job = None
    try:
      job = bq.query(
          sql,
          job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
      )
      res = job.result(page_size=5, timeout=20)  # Prevents hanging queries
    except (TimeoutError, concurrent.futures.TimeoutError):
      # result(timeout=) only stops us *waiting* -- the job keeps running and
      # billing. With 10 workers that can strand 10 jobs, so cancel explicitly.
      _cancel(job)
      # Slow is not wrong: the SQL passed the dry run, so TIMEOUT is final,
      # like TOO_BIG. Sending it to the LLM would rewrite a valid golden.
      return "TIMEOUT", "did not finish in 20s (dry run passed)", 0
    except Exception as e:  # pylint: disable=broad-except
      _cancel(job)
      if "bytes billed" in str(e).lower():
        return "TOO_BIG", f"exceeded the {MAX_GB_BILLED} GB cap while running", 0
      return "ERROR", _err_line(e, 200), 0
    n = res.total_rows or 0
    if n == 0:
      return "EMPTY", "0 rows (verify if expected)", 0
    if n == 1:
      # An aggregate without GROUP BY over 0 matching rows (e.g. an ungrounded
      # WHERE literal) returns 1 row where all columns are NULL. Hold it like
      # 0 rows unless KEEP_EMPTY_RESULTS=True.
      first = next(iter(res), None)
      vals = list(first.values() if hasattr(first, "values") else (first or ()))
      if vals and all(v is None for v in vals):
        return (
            "EMPTY",
            "1 row of all NULLs: filters may match no data (verify if expected)",
            1,
        )
    return "VALID", f"ok ({n} rows)", n

  def repair(question, sql, err):
    prompt = (
        "Fix this BigQuery SQL. Make it valid read-only GoogleSQL that "
        "answers the question. Check EVERY literal against the allowed "
        "values in the schema and correct any invented value. "
        "Do NOT add arbitrary filters just to force rows.\n"
        f"QUESTION: {question}\n"
        f"SCHEMA & REAL VALUES:\n{schema_text}\nFAILED SQL:\n{sql}\n"
        f"ERROR: {err}\nReturn ONLY the corrected SQL, no fences."
    )
    r = genai_client.models.generate_content(
        model=REPAIR_MODEL, contents=prompt
    )
    return strip_fences(r.text or "")

  # Only broken SQL goes to the LLM. TOO_BIG and TIMEOUT passed the dry run, so
  # they are final: rewriting them would change what the golden answers.
  REPAIRABLE = ("ERROR", "NON_SELECT")
  DRY_RUN_ONLY = ("TOO_BIG", "TIMEOUT")

  def process_pair(p):
    # Initialize defaults so every pair is guaranteed to have these keys
    p["repairs"] = 0
    p["rows"] = 0
    p["status"] = "NEEDS_REVIEW"
    p["reason"] = "UNKNOWN"
    p["detail"] = ""
    # The golden as first seen, so step 6 can show what repair changed. setdefault
    # keeps it when this cell is re-run on already-repaired pairs.
    p.setdefault("original_sql", p.get("golden_sql", ""))

    try:
      status, msg, n = validate(p.get("golden_sql", ""))
      while (
          MODE != "manual"
          and status in REPAIRABLE
          and p["repairs"] < MAX_REPAIRS
      ):
        p["repairs"] += 1
        try:
          fixed = repair(p.get("question", ""), p.get("golden_sql", ""), msg)
          if fixed:
            p["golden_sql"] = fixed
            status, msg, n = validate(fixed)
          else:
            break
        except Exception as e:
          msg = f"repair failed: {_err_line(e, 150)}"
          break

      keep = (
          status == "VALID"
          or (status == "EMPTY" and KEEP_EMPTY_RESULTS)
          or (status in DRY_RUN_ONLY and KEEP_DRY_RUN_ONLY)
      )
      p["status"] = "VALID" if keep else "NEEDS_REVIEW"
      p["reason"], p["rows"], p["detail"] = status, n, msg

    except Exception as e:
      p["status"] = "NEEDS_REVIEW"
      p["reason"] = "ERROR"
      p["detail"] = f"Unhandled error: {_err_line(e, 150)}"

    print(
        f"[{p.get('index', '?')}/{len(pairs)}] {p.get('status')}"
        f" ({p.get('reason')}) repairs={p.get('repairs', 0)}"
        f" rows={p.get('rows', 0)}",
        flush=True,
    )
    return p

  print(f"Validating {len(pairs)} pairs using 10 concurrent workers...")
  with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_pair, p) for p in pairs]
    for f in as_completed(futures):
      pass

  # Ensure pairs list stays sorted by original index
  pairs.sort(key=lambda x: x["index"])
  valid_count = sum(1 for p in pairs if p.get("status") == "VALID")
  empty_count = sum(1 for p in pairs if p.get("reason") == "EMPTY")
  print(f"\nDone. {valid_count}/{len(pairs)} pairs VALID.")
  if empty_count and not KEEP_EMPTY_RESULTS:
    print(
        f"NOTE: {empty_count} pair(s) returned 0 rows (or 1 row of all NULLs"
        " from an unmatched filter) and are held as NEEDS_REVIEW, so they will"
        " NOT be registered. Set KEEP_EMPTY_RESULTS=True in the config cell to"
        " keep them as intentional boundary cases."
    )
  dry_count = sum(1 for p in pairs if p.get("reason") in DRY_RUN_ONLY)
  if dry_count and not KEEP_DRY_RUN_ONLY:
    print(
        f"NOTE: {dry_count} pair(s) passed the dry run but were not executed"
        " (TOO_BIG: over MAX_GB_BILLED, or TIMEOUT: slower than 20s). They are"
        " held as NEEDS_REVIEW. Raise MAX_GB_BILLED, or set"
        " KEEP_DRY_RUN_ONLY=True to register them without an execution check."
    )
else:
  print(f"MODE={MODE}: validation skipped")

## 6. ⭐ Inspect — review before you register  ·  `cold` `seeded` `manual`

In [ ]:
# @title 6. Inspect the dataset
if MODE in ("cold", "seeded", "manual"):
  df = pd.DataFrame([
      {
          "#": p["index"],
          "status": p["status"],
          "reason": p.get("reason", ""),
          "rows": p["rows"],
          "repairs": p["repairs"],
          "question": p["question"],
      }
      for p in pairs
  ])
  pd.set_option("display.max_colwidth", 90)
  display(df[["#", "status", "reason", "rows", "repairs", "question"]])
  for p in pairs:
    print(f"\n[{p['index']}] {p['status']} ({p['detail']})")
    print("Q:  ", p["question"])
    orig = p.get("original_sql")
    if orig and orig != p["golden_sql"]:
      # Rewritten by the repair loop: show both so the change can be checked.
      print("ORIG:", " ".join(orig.split()))
      print("SQL: ", " ".join(p["golden_sql"].split()), "  <- repaired")
    else:
      print("SQL:", " ".join(p["golden_sql"].split()))
  print("\nKeep only rows you agree with. Edit `pairs` to drop/fix any.")
else:
  print(f"MODE={MODE}: nothing to inspect")

## 7. Register the EvaluationSet

Controlled by **`USE_GCS_IMPORT`** in the Configuration cell:
- **`USE_GCS_IMPORT = True` (`gcs_import`)**: writes the items as JSONL to `STAGING_BUCKET` and calls `evaluationSets:import`. Section B then sets `outputConfig.gcsDestination` on the run automatically.
- **`USE_GCS_IMPORT = False` (`inline_backoff`)**: creates the items one by one through the API (10 concurrent workers, automatic backoff on HTTP `429`/`5xx`), then calls `evaluationSets.create`. No Cloud Storage bucket needed.

All three dataset modes (`cold`, `seeded`, and `manual`) work with both methods.


In [ ]:
# @title 7. Register
# REGION (Vertex AI / Agent Platform), not DATA_AGENT_LOCATION -- the
# EvaluationSet is a regional resource.
parent = f"projects/{EVAL_PROJECT}/locations/{REGION}"
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
# Random suffix: two users (or two tabs) registering in the same second would
# otherwise write the same GCS prefix and pick each other's items.
set_display = f"{EVAL_SET_NAME}_{MODE}_{ts}_{uuid.uuid4().hex[:4]}"
EVAL_SET = None


def _poll_import(op_name, timeout=600):
  """Polls an evaluationSets:import long-running operation until completion."""
  t0 = time.time()
  while True:
    d = rest_get(op_name)
    if d.get("done"):
      if "error" in d:
        err = d["error"]
        if isinstance(err, dict) and err.get("code") == 13:
          # Usually means the Vertex AI (Agent Platform) Service Agent could not
          # read the bucket: the LRO reports only {'code': 13, 'message': 'INTERNAL'}.
          raise RuntimeError(
              f"evaluationSets:import failed with INTERNAL (code 13): {err}."
              f" The usual cause is that {EVAL_PROJECT}'s"
              " Vertex AI (Agent Platform) Service Agent"
              f" (service-<{EVAL_PROJECT} project"
              f" number>@{SERVICE_AGENT_DOMAIN}) cannot read"
              f" gs://{BARE_BUCKET}. Grant it roles/storage.objectAdmin on the"
              " bucket (prerequisite cell, step 4), wait a few minutes for IAM"
              " propagation, and re-run this cell."
          )
        raise RuntimeError(err)
      return d
    if time.time() - t0 > timeout:
      raise TimeoutError(f"import timed out after {timeout}s: {op_name}")
    time.sleep(3)


def _eval_set_from_op(op):
  """Pulls the EvaluationSet name out of a finished import LRO, or explains why not."""
  name = (op.get("response") or {}).get("evaluationSet")
  if not name:
    raise RuntimeError(
        "import LRO reported done but carried no evaluationSet. Raw response:\n"
        + json.dumps(op, indent=2)[:2000]
    )
  return name


def _import_set(body):
  op_init = rest_post(f"{parent}/evaluationSets:import", body)
  return _poll_import(op_init["name"])


def _upload(blob, **kw):
  """Uploads to STAGING_BUCKET; a 403 names the missing grant instead of a traceback."""
  try:
    if "filename" in kw:
      blob.upload_from_filename(
          kw["filename"], content_type="application/x-ndjson"
      )
    else:
      blob.upload_from_string(kw["data"], content_type="application/x-ndjson")
  except Exception as e:  # pylint: disable=broad-except
    hint = _iam_hint(e)
    if not hint and "403" in str(e):
      hint = (
          f"You need roles/storage.objectUser on gs://{BARE_BUCKET} to upload"
          " the import file (the prerequisite cell, 'What you need')."
      )
    if hint:
      raise RuntimeError(
          f"upload to gs://{BARE_BUCKET} failed: {_err_line(e, 200)}. {hint}"
      ) from e
    raise


if MODE in ("cold", "seeded", "manual"):
  _valid = [p for p in pairs if p["status"] == "VALID"]
  KEEP = _valid[:500]
  if len(_valid) > len(KEEP):
    print(
        f"WARN: {len(_valid)} VALID pairs truncated to {len(KEEP)} "
        "(EvaluationSet item cap)."
    )
  assert KEEP, (
      "No VALID pairs found to register! Check Step 5/6 validation results. If "
      "they are all EMPTY (0 rows) and that is expected, set "
      "KEEP_EMPTY_RESULTS=True in the config cell and re-run step 5."
  )
  print(f"Total valid pairs to register: {len(KEEP)}")

  if USE_GCS_IMPORT:
    print(
        "Registering via GCS Bulk Import (USE_GCS_IMPORT=True) ->"
        f" gs://{BARE_BUCKET}..."
    )
    jsonl_path = f"/tmp/{set_display}.jsonl"
    with open(jsonl_path, "w") as f:
      for p in KEEP:
        item = {
            "displayName": f"{EVAL_SET_NAME}_{p['index']}",
            "evaluationItemType": "REQUEST",
            "evaluationRequest": {
                "prompt": {"text": p["question"]},
                "goldenResponse": {
                    "candidate": "golden",
                    "text": p["golden_sql"],
                },
            },
        }
        f.write(json.dumps(item) + "\n")

    blob_path = f"eval_sets/{set_display}/items.jsonl"
    storage_client = storage.Client(project=EVAL_PROJECT, credentials=creds)
    bucket = storage_client.bucket(BARE_BUCKET)
    blob = bucket.blob(blob_path)
    _upload(blob, filename=jsonl_path)
    print(f"Uploaded JSONL to gs://{BARE_BUCKET}/{blob_path}")

    import_body = {
        "evaluation_set": {"display_name": set_display},
        "gcs_destination": {
            "output_uri_prefix": (
                f"gs://{BARE_BUCKET}/eval_sets/{set_display}/items/"
            )
        },
        "gcs_source": {
            "gcs_uri": f"gs://{BARE_BUCKET}/{blob_path}",
            "import_schema_config": {"data_format": "JSONL"},
        },
    }
    op = _import_set(import_body)
    EVAL_SET = _eval_set_from_op(op)
    print(f"Successfully imported EvaluationSet via GCS: {EVAL_SET}")

  else:
    print("Registering items inline (USE_GCS_IMPORT=False)...")

    def _register_item(p):
      payload = {
          "display_name": f"{EVAL_SET_NAME}_{p['index']}",
          "evaluation_item_type": "REQUEST",
          "evaluation_request": {
              "prompt": {"text": p["question"]},
              "golden_response": {
                  "candidate": "golden",
                  "text": p["golden_sql"],
              },
          },
      }
      return rest_post(f"{parent}/evaluationItems", payload)["name"]

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
      item_names = list(executor.map(_register_item, KEEP))

    create_set_resp = rest_post(
        f"{parent}/evaluationSets",
        {"display_name": set_display, "evaluation_items": item_names},
    )
    EVAL_SET = create_set_resp["name"]
    print(f"Successfully registered {len(item_names)} items inline: {EVAL_SET}")

print(f"\nEvaluationSet: {EVAL_SET}")

# Section B — Run + score

Submits an `EvaluationRun` for the data agent against the `EvaluationSet` from
Section A (wired automatically via `EVAL_SET`), polls to completion, and prints
the summary score + per-item detail. The run is created under `EVAL_PROJECT` and
points at the agent in `DATA_AGENT_PROJECT`.

In [ ]:
# @title 8. Evaluation Run & Scoring Helpers
TERMINAL = ("SUCCEEDED", "FAILED", "CANCELLED")
# poll() gives up after this many consecutive failed status checks (each one
# already retries 5 times) and says how to re-attach.
POLL_ERROR_BUDGET = 20


def parse_resource(name):
  """projects/{p}/locations/{loc}/... -> (project, location)."""
  parts = name.split("/")
  return parts[1], parts[3]


def require_set():
  """Validates EVALUATION_SET and returns (run_parent, set_name)."""
  if not EVALUATION_SET:
    raise ValueError(
        "Set EVALUATION_SET to a valid EvaluationSet resource name, e.g. "
        "projects/<proj>/locations/us-central1/evaluationSets/<id>."
    )
  proj, region = parse_resource(EVALUATION_SET)
  if region == "global":
    print(
        "WARNING: EVALUATION_SET is at locations/global. Regional runs "
        "(us-central1) are required to avoid initialization failures."
    )
  parent = f"projects/{proj}/locations/{region}"
  print("EvaluationSet:", EVALUATION_SET)
  print("Run parent   :", parent)
  return parent, EVALUATION_SET


def create_run(parent, set_name, name):
  # Reuse the single definition from the config cell so the agent location
  # cannot drift between here and step 1.
  assert SERVICE_ACCOUNT, (
      "Set SERVICE_ACCOUNT in the Configuration cell (required for data agent"
      " evaluation runs)."
  )
  data_agent = DATA_AGENT_RESOURCE
  body = {
      "displayName": name,
      "dataSource": {"evaluationSet": set_name},
      "evaluationConfig": {
          "metrics": [{
              "metric": METRIC,
              "predefinedMetricSpec": {"metricSpecName": METRIC},
          }]
      },
      "inferenceConfigs": {
          CANDIDATE: {
              "parallelism": 1,
              "agentRunConfig": {"dataAgentConfig": {"dataAgent": data_agent}},
          }
      },
      "serviceAccount": SERVICE_ACCOUNT,
  }
  # For GCS-backed sets, set outputConfig.gcsDestination. Note: NO trailing slash!
  if USE_GCS_IMPORT:
    body["evaluationConfig"]["outputConfig"] = {
        "gcsDestination": {
            "outputUriPrefix": f"gs://{BARE_BUCKET}/eval_runs/{name}_outputs"
        }
    }
  # Table access as SERVICE_ACCOUNT is not pre-checked here (it would need
  # serviceAccountTokenCreator for the caller). The service verifies it while
  # the run executes; poll() warns if the run stalls.
  print(json.dumps(body, indent=2))
  url = f"{_base()}/{parent}/evaluationRuns"
  for attempt in range(4):
    try:
      r = _eval_session.post(url, json=body, timeout=120)
    except requests.exceptions.RequestException as e:
      outcome = f"{type(e).__name__}: {_err_line(e, 200)}"
    else:
      if r.ok:
        return r.json()
      if r.status_code not in RETRYABLE:
        raise ApiError(
            "POST", f"{parent}/evaluationRuns", r.status_code, r.text
        )
      outcome = f"HTTP {r.status_code}"
    # Ambiguous: the run may exist even though we got no answer. Look it up by
    # display name before retrying, so a retry never starts a second run.
    time.sleep(2**attempt + random.uniform(0.1, 1.0))
    try:
      found = find_runs(parent, name)
    except Exception as e:  # pylint: disable=broad-except
      raise RuntimeError(
          f"create_run: POST failed ({outcome}) and listing runs to check"
          f" whether it was created also failed ({_err_line(e, 200)}). Look for"
          f" a run named {name!r} in the console before retrying."
      ) from e
    if found:
      print(
          f"create_run: POST failed ({outcome}) but {found[0]['name']} exists;"
          " using it."
      )
      return found[0]
    print(
        f"create_run: POST failed ({outcome}); no run named {name!r} yet,"
        " retrying."
    )
  raise RuntimeError(f"create_run: gave up after 4 attempts (last: {outcome}).")


def find_runs(parent, display_name):
  """Runs under parent with this display name (to adopt one after a lost response)."""
  filter_q = urllib.parse.quote(f'display_name="{display_name}"')
  runs = rest_get(f"{parent}/evaluationRuns?filter={filter_q}").get(
      "evaluationRuns", []
  )
  # Re-check here in case a server ignores the filter.
  return [r for r in runs if r.get("displayName") == display_name]


def delete_run(run_name):
  """Permanently deletes an EvaluationRun and its evaluation results.

  WARNING: Calling delete_run() permanently removes the EvaluationRun resource
  and all associated results. Use this only when you want to discard a stalled
  or unneeded run.
  """
  url = f"{_base()}/{run_name}"
  last = None
  for attempt in range(4):
    try:
      r = _eval_session.delete(url, timeout=120)
    except requests.exceptions.RequestException as e:
      last = f"{type(e).__name__}: {_err_line(e, 200)}"
    else:
      if r.ok:
        print(f"deleted {run_name} (and its results)")
        return r.json() if r.text else {}
      if r.status_code not in RETRYABLE:
        raise ApiError("DELETE", run_name, r.status_code, r.text)
      last = f"status={r.status_code} {r.text[:200]}"
    if attempt < 3:
      time.sleep(2**attempt + random.uniform(0.1, 1.0))
  raise RuntimeError(f"DELETE {run_name} failed after 4 attempts (last: {last})")


def get_run(name):
  return rest_get(name)


def poll(run_name, num_items=None):
  t0 = time.time()
  deadline = t0 + TIMEOUT_S
  n = 0
  last_state = None
  errors, last_err = 0, ""
  # Stall watchdog: the notebook cannot pre-check table access as SERVICE_ACCOUNT,
  # and a missing bigquery.dataViewer grant can keep INFERENCE busy for a long
  # time before the run fails. Warn when nothing changes for a while.
  state_since = t0
  # Scale the threshold with the dataset: the run queries the agent with
  # parallelism 1 (create_run), and a healthy agent turn takes well under a
  # minute, so _n_items * 2 min is a generous normal-case bound. Never below
  # INFERENCE_STALL_MIN.
  _n_items = (
      num_items
      if num_items is not None
      else len(globals().get("KEEP") or [])
  )
  stall_min = max(
      int(globals().get("INFERENCE_STALL_MIN", 20)),
      _n_items * 2 if _n_items else 0,
      1,
  )
  stall_s = stall_min * 60
  next_stall_warn = stall_s
  while True:
    try:
      run = get_run(run_name)
      errors = 0
    except ApiError as e:
      if e.status not in RETRYABLE:
        raise  # A 403 or 404 will not fix itself.
      errors, last_err = errors + 1, _err_line(e, 200)
    except (RuntimeError, requests.exceptions.RequestException) as e:
      errors, last_err = errors + 1, _err_line(e, 200)
    if errors:
      if errors >= POLL_ERROR_BUDGET or time.time() > deadline:
        raise RuntimeError(
            f"Lost contact with {run_name} after {errors} failed status checks"
            f" (last: {last_err}). The run is probably still going on the"
            f" server. To re-attach: terminal = poll({run_name!r}). To"
            " permanently delete the run and its results:"
            f" delete_run({run_name!r})."
        )
      print(
          f"  status check failed ({errors}/{POLL_ERROR_BUDGET}): {last_err}",
          flush=True,
      )
      time.sleep(POLL_INTERVAL_S)
      continue
    if not _n_items:
      src_set = (run.get("dataSource") or {}).get("evaluationSet")
      if src_set:
        try:
          _n_items = len(rest_get(src_set).get("evaluationItems", []))
          stall_min = max(
              int(globals().get("INFERENCE_STALL_MIN", 20)),
              _n_items * 2 if _n_items else 0,
              1,
          )
          stall_s = stall_min * 60
          next_stall_warn = stall_s
        except Exception:  # pylint: disable=broad-except
          _n_items = 0
    st = run.get("state", "STATE_UNSPECIFIED")
    elapsed = int(time.time() - t0)
    mm, ss = divmod(elapsed, 60)
    if st != last_state:
      state_since, next_stall_warn = time.time(), stall_s
    if st != last_state or n % 4 == 0:
      print(f"Poll #{n:3d} ({mm:02d}m{ss:02d}s) | state = {st}", flush=True)
      last_state = st
    if any(t in st for t in TERMINAL):
      print(f"\nRun completed in {mm:02d}m{ss:02d}s with final state: {st}")
      return run
    if time.time() - state_since >= next_stall_warn:
      stalled_min = int((time.time() - state_since) // 60)
      print(
          f"\n[STALL WARNING] {run_name} has been in state {st} for"
          f" {stalled_min} min with no state change (threshold {stall_min} min"
          f" for {_n_items or '?'} item(s)).\n  Most common cause:"
          " SERVICE_ACCOUNT cannot read one of the agent's BigQuery tables, or"
          " lacks roles/geminidataanalytics.dataAgentUser /"
          " cloudaicompanion.user (prerequisite cell, steps 2-3). Such runs"
          " can take a long time to fail. Fix the grant and start a new run."
          " To stop waiting and permanently delete this run and its results,"
          f" call\n  delete_run({run_name!r})\n  If the"
          " dataset is large, raise INFERENCE_STALL_MIN and keep waiting.",
          flush=True,
      )
      next_stall_warn += stall_s
    if time.time() > deadline:
      raise TimeoutError(
          f"{run_name} not done in {TIMEOUT_S}s (last={st}). It may still"
          f" finish: re-attach with poll({run_name!r}), or permanently delete"
          f" the run and its results with delete_run({run_name!r})."
      )
    n += 1
    time.sleep(POLL_INTERVAL_S)


def print_results(run):
  print("state:", run.get("state"))
  print("run  :", run.get("name"))
  if run.get("error"):
    print("error:", json.dumps(run["error"], indent=2))
  res = run.get("evaluationResults", {})
  if res.get("summaryMetrics"):
    print("\nsummaryMetrics:")
    print(json.dumps(res["summaryMetrics"], indent=2))
  if res.get("evaluationSet"):
    print("item-level results set:", res["evaluationSet"])


# Permission problems surface as free text in several fields; flag them loudly.
_IAM_RE = re.compile(r"access denied|permission|\b403\b", re.IGNORECASE)
# The agent's own answer is prose: a bare 403 there is usually a number
# ("1,403 trips"), so only the words count.
_IAM_ANSWER_RE = re.compile(r"access denied|permission", re.IGNORECASE)


def _wilson(k, n, z=1.96):
  """95% Wilson score interval for k successes in n trials."""
  if not n:
    return 0.0, 0.0
  p = k / n
  d = 1 + z * z / n
  c = (p + z * z / (2 * n)) / d
  h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
  return max(0.0, c - h), min(1.0, c + h)


def _prompt_text(resp):
  """Safely extracts request.prompt.text even when nested fields are null."""
  return (((resp or {}).get("request") or {}).get("prompt") or {}).get("text") or ""


def _item_status(resp):
  """Classifies one result item.

  Exactly one status per item:

  FETCH_ERROR   the result could not be read (e.g. GCS 403)
  UNGRADABLE    the metric could not grade the item (it drops out of the AVERAGE)
  NA/PASS/FAIL  graded; NA scores 1.0, like PASS
  AGENT_ERROR   calling the agent failed (error on the response or on the item)
  NOT_GRADED    the agent answered but no metric result was recorded
  NO_RESULT     nothing was recorded for this item
  """
  out = {
      "score": None,
      "detail": "",
      "iam_text": "",
      "question": _prompt_text(resp),
  }
  if "_fetch_error" in resp:
    out.update(
        status="FETCH_ERROR",
        detail=resp["_fetch_error"],
        iam_text=resp["_fetch_error"],
    )
    return out
  c_results = resp.get("candidateResults") or []
  cr = next((c for c in c_results if c.get("metric") == METRIC), {})
  if cr:
    add = cr.get("additionalResults", {}) or {}
    # NB: .get(k, default) only defaults on a MISSING key. The service sends
    # "explanation": null for ungradable items, so guard with `or`.
    expl = cr.get("explanation") or ""
    err = (cr.get("error") or {}).get("message", "") if cr.get("error") else ""
    answer = str(add.get("final_answer") or "")
    out.update(
        score=cr.get("score"),
        question=add.get("user_question") or out["question"],
        iam_text=" ".join(
            str(x)
            for x in (
                err,
                add.get("execution_error"),
                answer if _IAM_ANSWER_RE.search(answer) else "",
            )
            if x
        ),
    )
    # The rubric verdict is a bool, so NA is only visible in the text.
    is_na = expl.startswith("Verdict: NA") or any(
        (rv.get("reasoning") or "").startswith("[NA]")
        for rv in cr.get("rubricVerdicts", [])
    )
    if err and answer and not add.get("generated_sql"):
      # The metric requires a generated SQL query to compare against the golden
      # SQL. When the agent responds in prose only (for example, a clarifying
      # question, refusal, or schema description) without generating SQL, the
      # item is marked ungradable and excluded from the summary AVERAGE.
      out.update(
          status="UNGRADABLE",
          detail=f"agent answered without SQL (not graded): {answer[:100]}",
      )
    elif err:
      out.update(status="UNGRADABLE", detail=err)
    elif is_na:
      out.update(status="NA", detail=expl)
    elif out["score"] is None:
      out.update(status="UNGRADABLE", detail="no score and no error recorded")
    else:
      out.update(status="PASS" if out["score"] == 1.0 else "FAIL", detail=expl)
    return out
  # When an item fails while querying the agent (INFERENCE), candidateResults
  # is empty and the backend records the error on
  # request.candidateResponses[].error.
  cand_responses = (resp.get("request") or {}).get("candidateResponses") or []
  errs = []
  for cr_req in cand_responses:
    err_obj = cr_req.get("error")
    if err_obj:
      errs.append(
          err_obj.get("message", str(err_obj))
          if isinstance(err_obj, dict)
          else str(err_obj)
      )
  if errs:
    out.update(status="AGENT_ERROR", detail=errs[0], iam_text=" ".join(errs))
  elif cand_responses:
    out.update(
        status="NOT_GRADED",
        detail="agent answered, but no metric result was recorded",
    )
  elif resp.get("_item_error"):
    # Some inference failures are recorded only on the EvaluationItem, with no
    # agent response in the payload, e.g. code 13 "Internal error occurred."
    # for a table the service account cannot read.
    e = resp["_item_error"]
    msg = (e.get("message") or json.dumps(e)) if isinstance(e, dict) else str(e)
    code = e.get("code", "?") if isinstance(e, dict) else "?"
    out.update(
        status="AGENT_ERROR",
        detail=f"backend error {code}: {msg}",
        iam_text=msg,
        item_error=True,
    )
  else:
    out.update(status="NO_RESULT", detail="no result recorded for this item")
  return out


RESULT_EVAL_ITEMS = []
_LAST_RESULTS_SET_NAME = ""


def fetch_result_items(results_set_name):
  """Concurrently fetches all EvaluationItems (and GCS blobs) in a results EvaluationSet."""
  global RESULT_EVAL_ITEMS, _LAST_RESULTS_SET_NAME
  s = rest_get(results_set_name)
  items = s.get("evaluationItems", [])
  print(f"\n=== Fetching {len(items)} result item(s) concurrently... ===")
  st_client = storage.Client(project=EVAL_PROJECT, credentials=creds)

  def _fetch_one(item_name):
    # One unreadable item (GCS 403, a 5xx that outlived the retries) must not
    # sink the whole report; it becomes a FETCH_ERROR row instead.
    try:
      item = rest_get(item_name)
      resp = item.get("evaluationResponse") or {}
      gcs_uri = item.get("gcsUri", "")
      if not resp and gcs_uri.startswith("gs://"):
        bucket_name, blob_name = gcs_uri[5:].split("/", 1)
        d = json.loads(
            st_client.bucket(bucket_name).blob(blob_name).download_as_text()
        )
        resp = d.get("evaluationResponse", d)
      # Some failures are recorded only on the item itself, not in its payload.
      if item.get("error"):
        resp = dict(resp, _item_error=item["error"])
      return {
          "name": item.get("name", item_name),
          "displayName": item.get("displayName", ""),
          "gcsUri": gcs_uri,
          "error": item.get("error"),
          "evaluationItem": item,
          "evaluationResponse": resp,
      }
    except Exception as e:  # pylint: disable=broad-except
      err_msg = f"{type(e).__name__}: {_err_line(e, 200)}"
      return {
          "name": item_name,
          "displayName": "",
          "gcsUri": "",
          "error": {"message": err_msg},
          "evaluationItem": {"name": item_name},
          "evaluationResponse": {"_fetch_error": err_msg},
      }

  with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    RESULT_EVAL_ITEMS = list(executor.map(_fetch_one, items))
  _LAST_RESULTS_SET_NAME = results_set_name
  return RESULT_EVAL_ITEMS


def print_detailed_results(results_set_name, run=None):
  """Fetches item-level results; renders one row per item plus a summary.

  Pass the terminal run as `run` to check the local numbers against the
  backend summaryMetrics AVERAGE.
  """
  entries = fetch_result_items(results_set_name)
  payloads = [e["evaluationResponse"] for e in entries]

  rows, iam, item_errs = [], [], []
  for idx, resp in enumerate(payloads):
    st = _item_status(resp)
    rows.append({
        "#": idx + 1,
        "status": st["status"],
        "score": st["score"] if st["score"] is not None else "N/A",
        "question": st["question"][:80],
        "error_or_verdict": st["detail"][:120],
    })
    if st["iam_text"] and _IAM_RE.search(st["iam_text"]):
      iam.append(f"#{idx + 1} {st['status']}: {_err_line(st['iam_text'], 120)}")
    if st.get("item_error"):
      item_errs.append(idx + 1)

  if rows:
    df = pd.DataFrame(rows)
    display(df)

  print(f"\n=== Per-Item Metric Detail ({len(payloads)} item(s)) ===")
  for row, resp in zip(rows, payloads):
    q = _prompt_text(resp)
    c_results = resp.get("candidateResults", [])
    if c_results:
      for cr in c_results:
        add = cr.get("additionalResults", {}) or {}
        print("-" * 75)
        print(f"#{row['#']} [{row['status']}]")
        print("Q            :", add.get("user_question") or q)
        print(
            "metric/score :",
            cr.get("metric"),
            "->",
            cr.get("score"),
            "|",
            cr.get("explanation") or "",
        )
        if cr.get("error"):
          print("ungradable   :", cr["error"].get("message", cr["error"]))
        if add.get("generated_sql"):
          print("agent SQL    :", " ".join(add["generated_sql"].split()))
        if add.get("reference_sql"):
          print("reference SQL:", " ".join(add["reference_sql"].split()))
        if add.get("final_answer"):
          print("final answer :", add["final_answer"])
        if add.get("execution_error"):
          print("exec error   :", add["execution_error"])
        for rv in cr.get("rubricVerdicts", []):
          # verdict is a proto bool: false is omitted from the JSON, so None == FAIL.
          _reason = rv.get("reasoning", "") or ""
          _v = (
              "NA"
              if _reason.startswith("[NA]")
              else ("PASS" if rv.get("verdict") else "FAIL")
          )
          print("verdict      :", _v, "-", _reason)
    elif (resp.get("request") or {}).get("candidateResponses"):
      for cr_req in (resp.get("request") or {}).get("candidateResponses", []):
        err_obj = cr_req.get("error")
        err = (
            err_obj.get("message", str(err_obj))
            if isinstance(err_obj, dict)
            else str(err_obj or "No candidateResult produced")
        )
        print("-" * 75)
        print(f"#{row['#']} [{row['status']}]")
        print("Q            :", q)
        print("candidate    :", cr_req.get("candidate", CANDIDATE))
        print("agent error  :" if err_obj else "no result    :", err)
    else:
      print("-" * 75)
      print(f"#{row['#']} [{row['status']}]")
      print("Q            :", q or "(unknown)")
      print("detail       :", row["error_or_verdict"])

  counts = collections.Counter(r["status"] for r in rows)
  total = len(rows)
  graded = counts["PASS"] + counts["FAIL"] + counts["NA"]
  passed = counts["PASS"] + counts["NA"]
  print(f"\n=== Summary: {total} item(s) ===")
  print(
      "Status          : "
      + ", ".join(f"{k}={v}" for k, v in sorted(counts.items()))
  )
  if not total:
    return
  print(
      f"Coverage        : {graded}/{total} items graded ({graded / total:.1%})"
  )
  if graded:
    print(
        f"Graded accuracy : {passed}/{graded} = {passed / graded:.1%} "
        "(PASS + NA over graded items; what the backend AVERAGE reports)"
    )
  avg = (
      (
          ((run or {}).get("evaluationResults") or {}).get("summaryMetrics")
          or {}
      ).get("metrics")
      or {}
  ).get(f"{CANDIDATE}/{METRIC}/AVERAGE")
  if avg is not None and graded:
    if abs(avg - passed / graded) > 1e-6:
      print(
          f"MISMATCH        : backend AVERAGE is {avg:.4f} but local graded"
          f" accuracy is {passed / graded:.4f}; some results may not have been"
          " readable."
      )
    else:
      print(f"Backend AVERAGE : {avg:.4f} (matches)")
  lo, hi = _wilson(counts["PASS"], total)
  print(
      f"Strict accuracy : {counts['PASS']}/{total} ="
      f" {counts['PASS'] / total:.1%} (95% CI {lo:.0%}-{hi:.0%}); NA and every"
      " not-graded item count as misses"
  )
  if graded < total:
    print(
        "NOTE: not-graded items drop out of the backend AVERAGE, so it can look"
        " better than the agent is. Check Coverage before quoting a score."
    )
  if item_errs:
    shown = ", ".join(f"#{i}" for i in item_errs[:10]) + (
        " ..." if len(item_errs) > 10 else ""
    )
    print(
        f"NOTE: {len(item_errs)} item(s) failed inside the backend with no"
        f" agent response ({shown}). A table that SERVICE_ACCOUNT cannot read"
        " can fail this way, after a long INFERENCE phase. Check the service"
        " account's BigQuery read access on every table the agent uses."
    )
  if iam:
    print(
        f"\n[IAM WARNING] {len(iam)} item(s) mention a permission error, e.g."
        f" {iam[0]}"
    )
    _iam_all = " ".join(iam)
    if re.search(r"iam\.serviceAccounts\.getAccessToken", _iam_all):
      print(
          "  The Vertex AI (Agent Platform) Service Agent could not mint a"
          f" token for {SERVICE_ACCOUNT}. Check prerequisite step 2"
          f" ({EVAL_PROJECT}'s Vertex AI (Agent Platform) Service Agent needs"
          f" roles/iam.serviceAccountTokenCreator on {SERVICE_ACCOUNT})."
      )
    if re.search(
        r"User does not have permission to chat|dataAgents\.chat", _iam_all
    ):
      print(
          f"  {SERVICE_ACCOUNT} lacks roles/geminidataanalytics.dataAgentUser"
          f" on {DATA_AGENT_RESOURCE} (prerequisite cell, step 3b)."
      )
    if re.search(r"bigquery\.jobs\.create", _iam_all):
      print(
          f"  {SERVICE_ACCOUNT} lacks roles/bigquery.jobUser in"
          f" {DATA_AGENT_PROJECT} (prerequisite cell, step 3a)."
      )
    if re.search(r"permission to query table|Access Denied: Table", _iam_all):
      print(
          f"  {SERVICE_ACCOUNT} lacks roles/bigquery.dataViewer on one or more"
          f" of the agent's tables/datasets in {DATA_AGENT_PROJECT}"
          " (prerequisite cell, step 3c)."
      )
    if re.search(
        r"cloudaicompanion\.topics\.create|Failed to create conversation",
        _iam_all,
    ):
      print(
          "  Check prerequisite step 2 (Service Agent tokenCreator on"
          f" {SERVICE_ACCOUNT}) and step 3a ({SERVICE_ACCOUNT}"
          f" roles/cloudaicompanion.user in {DATA_AGENT_PROJECT})."
      )
    _su = next((h for h in (_iam_hint(x) for x in iam) if h), "")
    if _su:
      print(f"  {_su}")
    print(
        "  Agent or SQL errors: check SERVICE_ACCOUNT's BigQuery roles on the"
        " agent's tables (prerequisite cell, step 3c) and"
        " roles/geminidataanalytics.dataAgentUser on the agent (step 3b)."
        " FETCH_ERROR: check your own read access to STAGING_BUCKET. See the"
        " prerequisite cell."
    )


print("Evaluation helpers loaded.")

In [ ]:
# @title 9. Run + score
# EvaluationSet comes straight from Section A -- no copy-paste.
EVALUATION_SET = EVAL_SET
assert (
    EVALUATION_SET
), "Section A did not register an EvaluationSet (EVAL_SET is empty)."

parent, set_name = require_set()
# Printed before create_run: if the request is lost, find the run by this name.
RUN_DISPLAY_NAME = f"agent-eval-{uuid.uuid4().hex[:6]}"
print("run display name:", RUN_DISPLAY_NAME)
run = create_run(parent, set_name, RUN_DISPLAY_NAME)
print("run:", run["name"])

terminal = poll(run["name"])
print_results(terminal)

rs = terminal.get("evaluationResults", {}).get("evaluationSet")
if rs:
  print_detailed_results(rs, terminal)

In [ ]:
# @title 10. Fetch results EvaluationSet and EvaluationItems for an EvaluationRun
# Uses the EvaluationRun from Step 9 by default, or set EVAL_RUN_NAME to inspect any completed run:
# EVAL_RUN_NAME = f"projects/{EVAL_PROJECT}/locations/{REGION}/evaluationRuns/<EVAL_RUN_ID>"
EVAL_RUN_NAME = (
    run["name"]
    if "run" in globals() and isinstance(run, dict) and "name" in run
    else ""
)
assert EVAL_RUN_NAME, (
    "Run Step 9 first, or set EVAL_RUN_NAME to "
    'f"projects/{EVAL_PROJECT}/locations/{REGION}/evaluationRuns/<EVAL_RUN_ID>".'
)

# 1. Reuse the terminal EvaluationRun from Step 9 when inspecting the same run, or fetch it.
eval_run = (
    terminal
    if "terminal" in globals()
    and isinstance(terminal, dict)
    and terminal.get("name") == EVAL_RUN_NAME
    else get_run(EVAL_RUN_NAME)
)
results_eval_set_name = (eval_run.get("evaluationResults") or {}).get(
    "evaluationSet"
)
print("EvaluationRun        :", eval_run.get("name"))
print("Run state            :", eval_run.get("state"))
print("Results EvaluationSet:", results_eval_set_name)
assert results_eval_set_name, (
    f"No results evaluationSet found on {EVAL_RUN_NAME}"
    f" (state={eval_run.get('state')})."
)

# 2. Reuse RESULT_EVAL_ITEMS if Step 9 already fetched this results EvaluationSet,
#    otherwise fetch all EvaluationItems (and GCS blobs) concurrently via fetch_result_items().
if (
    globals().get("_LAST_RESULTS_SET_NAME") != results_eval_set_name
    or not globals().get("RESULT_EVAL_ITEMS")
):
  RESULT_EVAL_ITEMS = fetch_result_items(results_eval_set_name)

print(
    f"\nLoaded {len(RESULT_EVAL_ITEMS)} result EvaluationItem(s) into"
    " RESULT_EVAL_ITEMS."
)
for idx, entry in enumerate(RESULT_EVAL_ITEMS, 1):
  resp = entry["evaluationResponse"]
  st = _item_status(resp)
  c_results = resp.get("candidateResults") or []
  cr = next((c for c in c_results if c.get("metric") == METRIC), {})
  add = cr.get("additionalResults") or {}
  print("-" * 75)
  print(f"[{idx}] EvaluationItem : {entry['name']} ({st['status']})")
  if entry["displayName"]:
    print(f"    displayName    : {entry['displayName']}")
  if entry["gcsUri"]:
    print(f"    gcsUri         : {entry['gcsUri']}")
  print(f"    question       : {st['question']}")
  print(f"    score          : {st['score']}")
  if add.get("generated_sql"):
    print(f"    generated_sql  : {' '.join(add['generated_sql'].split())}")
  if add.get("reference_sql"):
    print(f"    reference_sql  : {' '.join(add['reference_sql'].split())}")
  if cr.get("explanation"):
    print(f"    explanation    : {cr['explanation']}")
  if st["status"] in ("UNGRADABLE", "AGENT_ERROR", "FETCH_ERROR") or entry["error"]:
    print(f"    detail/error   : {st['detail'] or entry['error']}")

In [ ]:
# @title 11. Optional: Run on a Subset of an Existing EvaluationSet
# Uncomment below to test a small subset (e.g. N=25 items) of an existing EvaluationSet:
#
# import uuid
# SRC_SET = f"projects/{EVAL_PROJECT}/locations/{REGION}/evaluationSets/<EVALUATION_SET_ID>"
# PARENT = f"projects/{EVAL_PROJECT}/locations/{REGION}"
# N = 25
# item_names = rest_get(SRC_SET).get("evaluationItems", [])[:N]
# small = rest_post(f"{PARENT}/evaluationSets",
#                   {"display_name": f"eval_subset_{N}_{uuid.uuid4().hex[:6]}",
#                    "evaluation_items": item_names})
# print("subset set:", small["name"], "items:", len(item_names))
# run = create_run(PARENT, small["name"], f"eval-subset-{uuid.uuid4().hex[:6]}")
# print("run:", run["name"])
# terminal = poll(run["name"])
# print_results(terminal)
# subset_results = terminal.get("evaluationResults", {}).get("evaluationSet")
# if subset_results:
#   print_detailed_results(subset_results, terminal)